# Brain Tumor MRI Classification (VGG16 Transfer Learning)

Classifies brain MRI slices into 4 classes: **glioma, meningioma, notumor, pituitary**.

- Method: VGG16 (ImageNet weights) as feature extractor, last convolutional layers fine-tuned
- Framework: TensorFlow / Keras
- Data: a public brain-tumor MRI dataset with `Training/` and `Testing/` folders, one sub-folder per class (add the source link here)

> For learning and experimentation only. This is **not** a medical device and must not be used for diagnosis.

**How to run:** set `DATA_DIR` in section 2 and run all cells from top to bottom (GPU runtime recommended).

## 1. Setup

In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Configuration

If you use Google Colab with Drive, uncomment the two mount lines. `DATA_DIR` must contain `Training/` and `Testing/`.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

DATA_DIR = "/content/brain_scans"        # <- change to your dataset folder
TRAIN_DIR = os.path.join(DATA_DIR, "Training")
TEST_DIR  = os.path.join(DATA_DIR, "Testing")

IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 15
VAL_SPLIT = 0.15
MODEL_PATH = "brain_tumor_vgg16.keras"

assert os.path.isdir(TRAIN_DIR) and os.path.isdir(TEST_DIR), "Check DATA_DIR: need Training/ and Testing/ folders"
for split in (TRAIN_DIR, TEST_DIR):
    print(split, {c: len(os.listdir(os.path.join(split, c))) for c in sorted(os.listdir(split))})

## 3. Data pipeline

- The same VGG16 `preprocess_input` is used for training, testing and single-image prediction.
- Augmentation is applied to **training data only**.
- The validation set is carved out of `Training/`; `Testing/` is kept untouched for the final evaluation.

In [ ]:
train_gen_cfg = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15, width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.1, horizontal_flip=True, fill_mode="nearest",
    validation_split=VAL_SPLIT,
)
eval_gen_cfg = ImageDataGenerator(preprocessing_function=preprocess_input, validation_split=VAL_SPLIT)
test_gen_cfg = ImageDataGenerator(preprocessing_function=preprocess_input)

common = dict(target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, class_mode="categorical", seed=SEED)
train_gen = train_gen_cfg.flow_from_directory(TRAIN_DIR, subset="training", shuffle=True, **common)
val_gen   = eval_gen_cfg.flow_from_directory(TRAIN_DIR, subset="validation", shuffle=False, **common)
test_gen  = test_gen_cfg.flow_from_directory(TEST_DIR, shuffle=False, **common)

class_names = [c for c, _ in sorted(train_gen.class_indices.items(), key=lambda kv: kv[1])]
print("Class order:", class_names)
assert class_names == [c for c, _ in sorted(test_gen.class_indices.items(), key=lambda kv: kv[1])]

## 4. Sample images

In [ ]:
x, y = next(iter(ImageDataGenerator(rescale=1/255).flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=12, class_mode="categorical", seed=SEED)))
plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x[i]); plt.title(class_names[int(np.argmax(y[i]))]); plt.axis("off")
plt.tight_layout(); plt.show()

## 5. Model

Stage 1 trains only the new classifier head on top of the frozen VGG16 base.

In [ ]:
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

model = Sequential([
    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    base_model,
    Flatten(),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(len(class_names), activation="softmax"),
])
model.compile(optimizer=Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

callbacks = [
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7),
]
history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)

## 6. Fine-tune the last VGG16 block

Stage 2 unfreezes the last 4 layers of the base model (`block5`) with a much smaller learning rate.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False

model.compile(optimizer=Adam(1e-5), loss="categorical_crossentropy", metrics=["accuracy"])
history_ft = model.fit(train_gen, validation_data=val_gen, epochs=5, callbacks=callbacks)

In [ ]:
acc  = history.history["accuracy"] + history_ft.history["accuracy"]
vacc = history.history["val_accuracy"] + history_ft.history["val_accuracy"]
loss = history.history["loss"] + history_ft.history["loss"]
vloss = history.history["val_loss"] + history_ft.history["val_loss"]
split_at = len(history.history["accuracy"]) - 1

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, (tr, va, name) in zip(ax, [(acc, vacc, "Accuracy"), (loss, vloss, "Loss")]):
    a.plot(tr, label="train"); a.plot(va, label="validation")
    a.axvline(split_at, color="gray", linestyle="--", label="fine-tuning starts")
    a.set_title(name); a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

## 7. Evaluate on the held-out test set

Metrics below come from `Testing/`, which was never used for training or validation.

In [ ]:
test_loss, test_acc = model.evaluate(test_gen, verbose=1)
print(f"Test accuracy: {test_acc:.4f} | Test loss: {test_loss:.4f}")

probs = model.predict(test_gen, verbose=0)
y_pred = np.argmax(probs, axis=1)
y_true = test_gen.classes

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix (test set)")
plt.tight_layout(); plt.show()

In [ ]:
y_bin = label_binarize(y_true, classes=np.arange(len(class_names)))
plt.figure(figsize=(8, 6))
for i, name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc(fpr, tpr):.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves (one-vs-rest, test set)"); plt.legend(loc="lower right"); plt.show()

## 8. Save the model and predict on a single image

In [ ]:
model.save(MODEL_PATH)
print("Saved:", MODEL_PATH)

def predict_image(image_path, model=model, class_names=class_names):
    """Predict the class of one MRI image (same preprocessing as training)."""
    img = load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = preprocess_input(np.expand_dims(img_to_array(img), axis=0))
    p = model.predict(arr, verbose=0)[0]
    idx = int(np.argmax(p))
    label = "No tumor" if class_names[idx] == "notumor" else f"Tumor: {class_names[idx]}"
    plt.imshow(load_img(image_path)); plt.axis("off")
    plt.title(f"{label} (confidence {p[idx]*100:.1f}%)"); plt.show()
    return class_names[idx], float(p[idx])

# Example (set a real path):
# predict_image(os.path.join(TEST_DIR, "meningioma", "<some_image>.jpg"))

## Results

_Fill this in after running the notebook:_ test accuracy, per-class precision/recall, and the confusion matrix.

## Limitations

- Trained and tested on one public dataset; performance on other scanners/hospitals is unknown.
- Slices from the same patient may appear in both train and test sets depending on the dataset, which can inflate scores.
- Research/education only, not for clinical use.